In [ ]:
import warnings
warnings.simplefilter("ignore")
import numpy as np
import pandas as pd 
from lightkurve import LightCurve
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
from lightkurve import search_lightcurve 
import lightkurve as lk

for att in ['axes.labelsize', 'axes.titlesize', 'legend.fontsize',
            'legend.fontsize', 'xtick.labelsize', 'ytick.labelsize']:
    plt.rcParams[att] = 15

search_result = search_lightcurve("AU Mic") 
print(search_result)
lc2min = search_result[4].download() 
lc2 = lc2min.remove_nans().remove_outliers()
lc2 = lc2[lc2.quality == 0]
lc2_m = lc2.normalize().remove_nans()

x2min = np.ascontiguousarray(lc2_m.time.value, dtype=np.float64) 
y2min = np.ascontiguousarray(lc2_m.flux, dtype=np.float64)
yerr2min = np.ascontiguousarray(lc2_m.flux_err, dtype=np.float64)
lcquality = np.ascontiguousarray(lc2_m.quality, dtype=np.float64)

In [1]:
# ===========================
# Centros dos trânsitos
# ===========================

# AU Mic b
centers_b = [
    3886.152646,
    3903.161623
]

# AU Mic c (tempos ajustados)
centers_c = [
    3888.24230,   # Época 135
    3907.10110    # Época 136
]

In [ ]:

# ============================================================
# PASSO 2: CARREGAR MÁSCARA DE FLARES E TRÂNSITOS (TXT EDITÁVEL)
# ============================================================
print("\n" + "="*60)
print("ENTRADA: Carregando intervalos de FLARES e TRÂNSITOS do TXT")
print("="*60)

t = np.array(lc2_m.time.value)
f = np.array(lc2_m.flux)

# Arquivo editável com colunas: t_ini, t_fim
txt_path = "intervalos_flares_transitos_editavel.txt"

try:
    # Lê como CSV simples (separado por vírgula), mas em arquivo .txt
    df_intervalos = pd.read_csv(txt_path, sep=",", comment="#")

    # Validação básica das colunas esperadas
    colunas_esperadas = {"t_ini", "t_fim"}
    if not colunas_esperadas.issubset(df_intervalos.columns):
        raise ValueError(f"Arquivo deve conter as colunas {colunas_esperadas}. Colunas encontradas: {set(df_intervalos.columns)}")

    # Converte para lista de pares [inicio, fim]
    mascara_flares_list = df_intervalos[["t_ini", "t_fim"]].dropna().values.tolist()

    print(f"✓ {len(mascara_flares_list)} intervalo(s) carregado(s) de '{txt_path}':")
    for i, (ini, fim) in enumerate(mascara_flares_list, start=1):
        print(f"  {i}. [{ini:.6f}, {fim:.6f}]")

except Exception as e:
    print(f"✗ Erro ao carregar '{txt_path}': {e}")
    print("→ Usando lista vazia.")
    mascara_flares_list = []

# Criar máscara booleana
mascara_flares = np.zeros(len(t), dtype=bool)
for ini, fim in mascara_flares_list:
    mascara_flares |= (t >= ini) & (t <= fim)

mask_good_flares = ~mascara_flares

print(f"\nMáscara criada:")
print(f"  Pontos excluídos (flares/trânsitos): {mascara_flares.sum()}")
print(f"  Pontos para ajuste: {mask_good_flares.sum()}")



ENTRADA: Carregando intervalos de FLARES e TRÂNSITOS do TXT


NameError: name 'np' is not defined

: 

In [ ]:

# ============================================================
# PASSO 2: CARREGAR MÁSCARA DE FLARES E TRÂNSITOS (TXT EDITÁVEL)
# ============================================================
print("\n" + "="*60)
print("ENTRADA: Carregando intervalos de FLARES e TRÂNSITOS do TXT")
print("="*60)

t = np.array(lc2_m.time.value)
f = np.array(lc2_m.flux)

# Arquivo editável com colunas: t_ini, t_fim
txt_path = "intervalos_flares_transitos_editavel.txt"

try:
    # Lê como CSV simples (separado por vírgula), mas em arquivo .txt
    df_intervalos = pd.read_csv(txt_path, sep=",", comment="#")

    # Validação básica das colunas esperadas
    colunas_esperadas = {"t_ini", "t_fim"}
    if not colunas_esperadas.issubset(df_intervalos.columns):
        raise ValueError(f"Arquivo deve conter as colunas {colunas_esperadas}. Colunas encontradas: {set(df_intervalos.columns)}")

    # Converte para lista de pares [inicio, fim]
    mascara_flares_list = df_intervalos[["t_ini", "t_fim"]].dropna().values.tolist()

    print(f"✓ {len(mascara_flares_list)} intervalo(s) carregado(s) de '{txt_path}':")
    for i, (ini, fim) in enumerate(mascara_flares_list, start=1):
        print(f"  {i}. [{ini:.6f}, {fim:.6f}]")

except Exception as e:
    print(f"✗ Erro ao carregar '{txt_path}': {e}")
    print("→ Usando lista vazia.")
    mascara_flares_list = []

# Criar máscara booleana
mascara_flares = np.zeros(len(t), dtype=bool)
for ini, fim in mascara_flares_list:
    mascara_flares |= (t >= ini) & (t <= fim)

mask_good_flares = ~mascara_flares

print(f"\nMáscara criada:")
print(f"  Pontos excluídos (flares/trânsitos): {mascara_flares.sum()}")
print(f"  Pontos para ajuste: {mask_good_flares.sum()}")



ENTRADA: Carregando intervalos de FLARES e TRÂNSITOS do TXT
✓ 26 intervalo(s) carregado(s) de 'intervalos_flares_transitos_editavel.txt':
  1. [3883.239461, 3883.334868]
  2. [3884.751868, 3884.843937]
  3. [3885.024950, 3885.109542]
  4. [3885.549959, 3885.617899]
  5. [3886.130183, 3886.411659]
  6. [3888.081298, 3888.115210]
  7. [3888.784350, 3888.805991]
  8. [3889.179949, 3889.251286]
  9. [3889.826802, 3889.854535]
  10. [3891.013277, 3891.076262]
  11. [3891.741181, 3891.795356]
  12. [3891.844004, 3891.908792]
  13. [3891.950028, 3892.001562]
  14. [3892.631057, 3892.697367]
  15. [3892.812488, 3892.900441]
  16. [3893.017865, 3893.101673]
  17. [3893.403291, 3893.458281]
  18. [3896.687407, 3896.849926]
  19. [3897.567644, 3897.779646]
  20. [3899.153300, 3899.218797]
  21. [3899.542373, 3899.616046]
  22. [3900.664900, 3900.748400]
  23. [3900.906700, 3900.408000]
  24. [3904.512267, 3904.614076]
  25. [3904.782061, 3904.847219]
  26. [3905.340994, 3905.482584]

Máscara cri

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# PASSO 3: AJUSTE ROTACIONAL BASEADO EM CADÊNCIA + AJUSTES MANUAIS
# ============================================================

print("\n" + "="*60)
print("AJUSTE: Modelo Rotacional Físico (Global) + Correções Locais e Lineares")
print("="*60)

t = np.array(lc2_m.time.value)
f = np.array(lc2_m.flux)

# ============================================================
# FUNÇÕES PRINCIPAIS
# ============================================================

def gerar_modelo_rotacional_cadencia(t, f, mask_good_flares, cadencia_s=120, janela_horas=10, sigma_clip_val=1.5):
    from scipy.ndimage import gaussian_filter1d
    from astropy.stats import sigma_clip
    
    pontos_por_hora = 3600 / cadencia_s
    janela_pontos = int(janela_horas * pontos_por_hora)
    
    sigma_pontos = janela_pontos / 8
    
    gap_minimo_minutos = 20
    limite_gap = gap_minimo_minutos / (24 * 60)
    quebras = list(np.where(np.diff(t) > limite_gap)[0] + 1)
    seg_inicios = [0] + quebras
    seg_fins    = quebras + [len(t)]
    
    modelo_final = np.zeros_like(f)
    
    print(f"\n[Filtro Base Global] Janela: {janela_horas}h | Sigma Clip: {sigma_clip_val}")
    
    for i0, i1 in zip(seg_inicios, seg_fins):
        t_seg = t[i0:i1]
        f_seg = f[i0:i1]
        mask_seg = mask_good_flares[i0:i1]
        
        if len(t_seg) < janela_pontos / 4:
            modelo_final[i0:i1] = np.nanmedian(f_seg)
            continue
            
        f_limpo = np.copy(f_seg)
        
        for iteracao in range(10):
            temp_smooth = gaussian_filter1d(f_limpo, sigma=sigma_pontos, mode='nearest')
            residuo = f_seg - temp_smooth
            
            clipped = sigma_clip(
                residuo, 
                sigma_lower=10.0,
                sigma_upper=sigma_clip_val, 
                maxiters=1, 
                cenfunc='median', 
                stdfunc='mad_std'
            )            
            f_limpo[clipped.mask] = temp_smooth[clipped.mask]
            f_limpo[~mask_seg] = temp_smooth[~mask_seg] 
            
        modelo_final[i0:i1] = gaussian_filter1d(f_limpo, sigma=sigma_pontos, mode='nearest')
        
    return modelo_final

def ajustar_trecho_especifico(t, f, mask_good_flares, t_inicio, t_fim, 
                              cadencia_s=120, janela_horas=5, 
                              sigma_upper=2.0, sigma_lower=3.0, iteracoes=6):
    """
    Usa a EXATA MESMA LÓGICA do modelo global, mas com parâmetros finos 
    aplicados apenas a um recorte, usando margem de segurança.
    """
    from scipy.ndimage import gaussian_filter1d
    from astropy.stats import sigma_clip
    
    buffer_dias = (janela_horas * 1.5) / 24.0 
    idx_calc = np.where((t >= t_inicio - buffer_dias) & (t <= t_fim + buffer_dias))[0]
    idx_alvo = np.where((t >= t_inicio) & (t <= t_fim))[0]
    
    if len(idx_alvo) == 0:
        return idx_alvo, np.array([])

    t_calc = t[idx_calc]
    f_calc = f[idx_calc]
    mask_calc = mask_good_flares[idx_calc]
    
    pontos_por_hora = 3600 / cadencia_s
    sigma_pontos = (janela_horas * pontos_por_hora) / 8
    
    f_limpo = np.copy(f_calc)
    
    for _ in range(iteracoes):
        temp_smooth = gaussian_filter1d(f_limpo, sigma=sigma_pontos, mode='reflect')
        residuo = f_calc - temp_smooth
        
        clipped = sigma_clip(residuo, sigma_lower=sigma_lower, sigma_upper=sigma_upper, 
                             maxiters=1, cenfunc='median', stdfunc='mad_std')
        
        mascara_combinada = (~clipped.mask) & mask_calc
        t_bons = t_calc[mascara_combinada]
        f_bons = f_limpo[mascara_combinada]
        
        if len(t_bons) > 2:
            f_limpo = np.interp(t_calc, t_bons, f_bons)
        else:
            f_limpo[clipped.mask] = temp_smooth[clipped.mask]
            
    modelo_calc = gaussian_filter1d(f_limpo, sigma=sigma_pontos, mode='nearest')
    
    inicio_corte = np.where(idx_calc == idx_alvo[0])[0][0]
    fim_corte = np.where(idx_calc == idx_alvo[-1])[0][0] + 1
    
    return idx_alvo, modelo_calc[inicio_corte:fim_corte]

# --- NOVA FUNÇÃO PARA FIT LINEAR ---
def ajustar_trecho_linear(t, f, mask_good_flares, t_inicio, t_fim, sigma_upper=2.0, sigma_lower=3.0, offset_y=0.0, tilt=0.0):
    """
    Traça uma reta no trecho e permite ajustes manuais de altura (offset) e inclinação (tilt).
    """
    from astropy.stats import sigma_clip
    
    idx_alvo = np.where((t >= t_inicio) & (t <= t_fim))[0]
    if len(idx_alvo) == 0:
        return idx_alvo, np.array([])

    t_alvo = t[idx_alvo]
    f_alvo = f[idx_alvo]
    mask_alvo = mask_good_flares[idx_alvo]

    t_limpo = t_alvo[mask_alvo]
    f_limpo = f_alvo[mask_alvo]

    clipped = sigma_clip(f_limpo, sigma_lower=sigma_lower, sigma_upper=sigma_upper, maxiters=2)
    bons = ~clipped.mask

    if np.sum(bons) > 2:
        coefs = np.polyfit(t_limpo[bons], f_limpo[bons], deg=1)
        
        # --- CONTROLE MANUAL DA RETA ---
        inclinacao_original = coefs[0]
        nova_inclinacao = inclinacao_original + tilt
        
        # Ponto de giro (centro do trecho no eixo x)
        t_centro = np.mean(t_alvo)
        y_centro = np.polyval(coefs, t_centro)
        
        # Equação da reta rotacionada e transladada: y = m*(x - x0) + y0 + offset
        modelo_linear = nova_inclinacao * (t_alvo - t_centro) + y_centro + offset_y
        
    else:
        modelo_linear = (np.ones_like(t_alvo) * np.nanmedian(f_alvo)) + offset_y

    return idx_alvo, modelo_linear
# -----------------------------------
# -----------------------------------

def costurar_bordas(t, modelo, bordas, tamanho_janela=25, sigma_gauss=0):
    from scipy.ndimage import gaussian_filter1d
    modelo_costurado = np.copy(modelo).astype(float)
    n = len(t)
    
    for borda_idx in sorted(bordas):
        inicio = max(0, borda_idx - tamanho_janela)
        fim = min(n - 1, borda_idx + tamanho_janela)
        # Reduzindo a restrição de tamanho mínimo já que temos menos pontos
        if (fim - inicio) < 10: continue
            
        meia_zona = (fim - inicio) // 4
        fim_esq = max(inicio + 2, borda_idx - meia_zona)
        ini_dir = min(fim - 2, borda_idx + meia_zona)
        
        idx_esq = np.arange(inicio, fim_esq)
        idx_dir = np.arange(ini_dir, fim + 1)
        if len(idx_esq) < 5 or len(idx_dir) < 5: continue
            
        t_centro = t[borda_idx]
        poly_esq = np.polyfit(t[idx_esq] - t_centro, modelo_costurado[idx_esq], deg=2)
        poly_dir = np.polyfit(t[idx_dir] - t_centro, modelo_costurado[idx_dir], deg=2)
        
        zona_miolo = np.arange(fim_esq, ini_dir + 1)
        t_miolo = t[zona_miolo] - t_centro
        
        pred_esq = np.polyval(poly_esq, t_miolo)
        pred_dir = np.polyval(poly_dir, t_miolo)
        
        x_norm = (t[zona_miolo] - t[fim_esq]) / (t[ini_dir] - t[fim_esq] + 1e-10)
        x_norm = np.clip(x_norm, 0, 1)
        peso = 3 * x_norm**2 - 2 * x_norm**3 
        
        modelo_costurado[zona_miolo] = (1 - peso) * pred_esq + peso * pred_dir
        
    if sigma_gauss > 0:
        dt_mediano = np.median(np.diff(t))
        quebras = list(np.where(np.diff(t) > dt_mediano * 5)[0] + 1)
        for i0, i1 in zip([0] + quebras, quebras + [n]):
            seg = modelo_costurado[i0:i1]
            if len(seg) > 1:
                modelo_costurado[i0:i1] = gaussian_filter1d(seg, sigma=sigma_gauss)
                
    return modelo_costurado

def costurar_com_gaps(t, modelo, bordas, limite_gap_fator=5, tamanho_janela=5, sigma_gauss=3):
    from scipy.ndimage import gaussian_filter1d
    dt_mediano = np.median(np.diff(t))
    limite_gap = dt_mediano * limite_gap_fator
    quebras = list(np.where(np.diff(t) > limite_gap)[0] + 1)
    seg_inicios = [0] + quebras
    seg_fins    = quebras + [len(t)]

    modelo_costurado = costurar_bordas(
        t, modelo, bordas,
        tamanho_janela=tamanho_janela,
        sigma_gauss=0 
    )

    if sigma_gauss > 0:
        for i0, i1 in zip(seg_inicios, seg_fins):
            seg = modelo_costurado[i0:i1]
            if len(seg) > 1:
                modelo_costurado[i0:i1] = gaussian_filter1d(seg, sigma=sigma_gauss)

    return modelo_costurado

# ============================================================
# GERANDO A BASE OFICIAL GLOBAL
# ============================================================

# Modelo global atualizado para 120s
modelo_manchas = gerar_modelo_rotacional_cadencia(
    t, f, mask_good_flares, 
    cadencia_s=120, 
    janela_horas=10, 
    sigma_clip_val=1.8
)

bordas_indices = set()

# ============================================================
# REGIÕES DE AJUSTE FINO E LINEAR
# ============================================================

# 1. Ajustes locais via Gaussiana [t_inicio, t_fim, janela_horas, sigma_upper, sigma_lower, iteracoes]
regioes_ajuste_local = [
    [3884.6271, 3884.9503, 10.0, 1.2, 3.0, 12],
    [3885.9927, 3886.4632, 10.0, 5, 5.0, 12],
    #[3900.1457, 3901.0141, 10.0, 1.5, 1.0, 10],
    [3900.1457, 3901.0141, 10.0, 0.8, 0.8, 10],
    [3902.9503, 3903.4152, 10.0, 0.8, 0.8, 10]

]

# 2. Ajustes locais via Reta Linear [t_inicio, t_fim, sigma_upper, sigma_lower]
regioes_ajuste_linear = [
   [3906.9180,3907.1300, 15.5, 9.0, 0.0001, 0.0005]
]

# Aplicando os ajustes Gaussianos
for ini, fim, janela_h, sig_up, sig_low, iters in regioes_ajuste_local:
    idx = (t >= ini) & (t <= fim)
    if not np.any(idx): continue
    
    idx_alvo, mod_local = ajustar_trecho_especifico(
        t, f, mask_good_flares, t_inicio=ini, t_fim=fim, 
        cadencia_s=120, # Atualizado aqui também
        janela_horas=janela_h, sigma_upper=sig_up, sigma_lower=sig_low, iteracoes=iters
    )
    
    if len(idx_alvo) > 0:
        modelo_manchas[idx_alvo] = mod_local
        bordas_indices.add(idx_alvo[0])
        bordas_indices.add(idx_alvo[-1])

# Aplicando os ajustes Lineares
for ini, fim, sig_up, sig_low, off_y, tilt_val in regioes_ajuste_linear:
    idx = (t >= ini) & (t <= fim)
    if not np.any(idx): continue
    
    idx_alvo, mod_linear = ajustar_trecho_linear(
        t, f, mask_good_flares, t_inicio=ini, t_fim=fim, 
        sigma_upper=sig_up, sigma_lower=sig_low, 
        offset_y=off_y, tilt=tilt_val 
    )
    
    if len(idx_alvo) > 0:
        modelo_manchas[idx_alvo] = mod_linear
        bordas_indices.add(idx_alvo[0])
        bordas_indices.add(idx_alvo[-1])

# ============================================================
# COSTURA FINAL DAS REGIÕES LOCAIS E LINEARES
# ============================================================

if len(bordas_indices) > 0:
    print(f"\nCosturando {len(bordas_indices)} bordas de ajustes locais e lineares...")
    modelo_manchas_suave = costurar_bordas(
        t, modelo_manchas, sorted(list(bordas_indices)),
        tamanho_janela=25, # Reduzido (era 150), proporcional à nova cadência
        sigma_gauss=3      # Reduzido (era 20), proporcional à nova cadência
    )
else:
    print("\nNenhum ajuste local aplicado. Usando o modelo global liso.")
    from scipy.ndimage import gaussian_filter1d
    modelo_manchas_suave = np.copy(modelo_manchas)
    
    dt_mediano = np.median(np.diff(t))
    quebras = list(np.where(np.diff(t) > dt_mediano * 5)[0] + 1)
    for i0, i1 in zip([0] + quebras, quebras + [len(t)]):
        if i1 - i0 > 1:
            modelo_manchas_suave[i0:i1] = gaussian_filter1d(modelo_manchas_suave[i0:i1], sigma=3) # Atualizado aqui também


# ============================================================
# RESÍDUO E GRÁFICOS
# ============================================================

%matplotlib qt

import matplotlib.pyplot as plt

residual_manchas = f / modelo_manchas_suave
print("\nOK — Ajuste concluído.")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

ax1.plot(t, f, 'k.-', ms=1.5, lw=0.5, alpha=0.6, label='Dados')
ax1.plot(t, modelo_manchas_suave, color='red', lw=2.2, label='Modelo Final')

# Pinta de azul as regiões do ajuste local (gaussiano)
for ini, fim, *_ in regioes_ajuste_local:
    ax1.axvspan(ini, fim, color='dodgerblue', alpha=0.15, label='Ajuste Local Gaussiano' if ini == regioes_ajuste_local[0][0] else "")

# Pinta de laranja a região do ajuste linear
if len(regioes_ajuste_linear) > 0:
    for ini, fim, *_ in regioes_ajuste_linear:
        ax1.axvspan(ini, fim, color='orange', alpha=0.25, label='Ajuste Linear')

ax1.set_ylabel("Fluxo")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(t, residual_manchas, 'b.-', ms=1.5, lw=0.5, alpha=0.7)
ax2.axhline(1, ls='--', alpha=0.5)

for ini, fim, *_ in regioes_ajuste_local:
    ax2.axvspan(ini, fim, color='dodgerblue', alpha=0.15)
for ini, fim, *_ in regioes_ajuste_linear:
    ax2.axvspan(ini, fim, color='orange', alpha=0.25)

ax2.set_ylabel("Residual")
ax2.set_xlabel("Tempo [BTJD]")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()


AJUSTE: Modelo Rotacional Físico (Global) + Correções Locais e Lineares

[Filtro Base Global] Janela: 10h | Sigma Clip: 1.8

Costurando 10 bordas de ajustes locais e lineares...

OK — Ajuste concluído.


In [ ]:
from astropy.stats import sigma_clipped_stats
from scipy.signal import find_peaks
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# PARÂMETROS AU Mic d
# ============================================================
T0_d   = 2458333.3211 - 2457000
P_d    = 12.73596
dur_d  = 0.12   # horas convertidas aprox (2.9h)

# ============================================================
# PARÂMETROS AU Mic b
# ============================================================
T0_b   = 2458330.3905 - 2457000
P_b    = 8.462999
dur_b  = 0.08   # ~2 horas

# ============================================================
# PARÂMETROS AU Mic c
# ============================================================
T0_c   = 2458332.3644 - 2457000
P_c    = 18.792819
dur_c  = 0.10

def transit_times(T0, P, t_min, t_max):
    n_min = int(np.ceil((t_min - T0) / P))
    n_max = int(np.floor((t_max - T0) / P))
    return [T0 + n * P for n in range(n_min, n_max + 1)]

# ============================================================
# TRÂNSITOS
# ============================================================
centers_d = transit_times(T0_d, P_d, t.min(), t.max())
centers_b = transit_times(T0_b, P_b, t.min(), t.max())
centers_c = transit_times(T0_c, P_c, t.min(), t.max())

print(f"Trânsitos AU Mic d: {len(centers_d)}")

# ============================================================
# FILTRAGEM
# ============================================================
x_detrend = []
y_detrend = []

for j in range(len(mask_good_flares)):
    if mask_good_flares[j]:
        y_detrend.append(residual_manchas[j])
        x_detrend.append(t[j])

x_detrend = np.array(x_detrend)
y_detrend = np.array(y_detrend)

# ============================================================
# ESTATÍSTICA
# ============================================================
y_mean = np.mean(y_detrend)
desvio = np.std(y_detrend)

med2 = y_mean + 2 * desvio
med2_inf = y_mean - 2 * desvio

# ============================================================
# SIGMA CLIPPING
# ============================================================
y_m, _, desvio2 = sigma_clipped_stats(y_detrend, sigma=3, maxiters=5)

med3 = y_m + 3 * desvio2

# ============================================================
# PICOS
# ============================================================
peaks, _ = find_peaks(y_detrend, height=med3)

# ============================================================
# FIGURA
# ============================================================
fig, ax = plt.subplots(1, 1, figsize=(14, 6))

ax.plot(t, residual_manchas, 'k.-', ms=1.5, lw=0.5, alpha=0.5)

ax.axhline(y_m, color='red', label='Média')
ax.axhline(med3, color='orange', label='+3σ')

ax.plot(x_detrend[peaks], y_detrend[peaks],
        'x', color='deeppink', label='Picos')

# =========================
# MÁSCARA SEGURA (se existir)
# =========================
for i, tc in enumerate(centers_b):
    if dur_b is not None:
        ax.axvspan(tc-dur_b/2, tc+dur_b/2,
                   color='royalblue', alpha=0.18,
                   label='AU Mic b' if i == 0 else None)
    ax.axvline(tc, color='royalblue', lw=1)

for i, tc in enumerate(centers_c):
    if dur_c is not None:
        ax.axvspan(tc-dur_c/2, tc+dur_c/2,
                   color='seagreen', alpha=0.18,
                   label='AU Mic c' if i == 0 else None)
    ax.axvline(tc, color='seagreen', lw=1)

for i, tc in enumerate(centers_d):
    ax.axvline(tc, color='purple', lw=1.2,
               label='AU Mic d' if i == 0 else None)

ax.legend()
ax.set_xlabel("BTJD")
ax.set_ylabel("Fluxo residual")
ax.set_title("AU Mic – Trânsitos + flares + picos")

plt.tight_layout()
plt.show()

Trânsitos AU Mic d: 2


In [ ]:
from astropy.stats import sigma_clipped_stats
from scipy.signal import find_peaks
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# CARREGAMENTO DA MÁSCARA (flares_transit_1.txt)
# ============================================================
try:
    df_nova_mascara = pd.read_csv('flares_transit_1.txt')
    print("✅ Arquivo 'flares_transit_1.txt' carregado com sucesso!")
except FileNotFoundError:
    print("⚠️ Arquivo 'flares_transit_1.txt' não encontrado! Criando DataFrame vazio.")
    df_nova_mascara = pd.DataFrame(columns=['Inicio', 'Fim'])

# ============================================================
# PARÂMETROS AU Mic d E CÁLCULO DOS TRÂNSITOS
# ============================================================
T0_d   = 2458333.3211 - 2457000
P_d    = 12.73596
dur_d  = None

def transit_times(T0, P, t_min, t_max):
    n_min = int(np.ceil((t_min - T0) / P))
    n_max = int(np.floor((t_max - T0) / P))
    return [T0 + n * P for n in range(n_min, n_max + 1)]

# Assumindo que o array 't' já existe
centers_d = transit_times(T0_d, P_d, t.min(), t.max())

print(f"Trânsitos AU Mic d: {len(centers_d)}")
for tc in centers_d:
    print(f"  BTJD = {tc:.5f}")

# ============================================================
# FILTRAGEM DOS PONTOS BONS (Apenas com a máscara do TXT)
# ============================================================
x_detrend = []
y_detrend = []

for j in range(len(t)):
    tempo_atual = t[j]
    is_good = True
    
    # Verifica se o ponto atual cai dentro de alguma região do txt
    for _, linha in df_nova_mascara.iterrows():
        if linha['Inicio'] <= tempo_atual <= linha['Fim']:
            is_good = False  # Ponto cai na máscara, então é descartado
            break
                
    # Salva apenas se o ponto for bom (fora da máscara)
    if is_good:
        y_detrend.append(residual_manchas[j])
        x_detrend.append(t[j])

x_detrend = np.array(x_detrend)
y_detrend = np.array(y_detrend)

print(f"\nPontos totais:     {len(t)}")
print(f"Pontos bons:       {len(y_detrend)}")
print(f"Pontos mascarados: {len(t)-len(y_detrend)}")

# ============================================================
# MÉNIA E DESVIO INICIAIS
# ============================================================
y_mean = np.mean(y_detrend)
desvio = np.std(y_detrend)

med_desv2 = y_mean + 2 * desvio
med_desv3 = y_mean - 2 * desvio

print(f"\n--- 1ª iteração ---")
print(f"Média:                {y_mean:.6f}")
print(f"Desvio padrão:        {desvio:.6f}")
print(f"Corte superior (+2σ): {med_desv2:.6f}")
print(f"Corte inferior (-2σ): {med_desv3:.6f}")

# ============================================================
# CORTE ±2σ
# ============================================================
xdefinitivo = []
ydefinitivo = []

for j in range(len(y_detrend)):
    if med_desv3 < y_detrend[j] < med_desv2:
        ydefinitivo.append(y_detrend[j])
        xdefinitivo.append(x_detrend[j])

xdefinitivo = np.array(xdefinitivo)
ydefinitivo = np.array(ydefinitivo)

print(f"\nPontos após corte ±2σ: {len(ydefinitivo)}")

# ============================================================
# SIGMA CLIPPING
# ============================================================
y_m, _, desvio2 = sigma_clipped_stats(ydefinitivo, sigma=3, maxiters=5)

med2 = y_m + 3 * desvio2
med2_inf = y_m - 3 * desvio2

print(f"\n--- 2ª iteração (sigma-clipping 3σ iterativo) ---")
print(f"Média clipped:  {y_m:.6f}")
print(f"Desvio padrão:  {desvio2:.6f}")
print(f"Limiar +3σ:     {med2:.6f}")
print(f"Limiar -3σ:     {med2_inf:.6f}")

# ============================================================
# DETECÇÃO DOS PICOS
# ============================================================
peaks, properties = find_peaks(y_detrend, height=med2)

print(f"\nNúmero de picos encontrados: {len(peaks)}")
for i in peaks:
    print(f"Tempo = {x_detrend[i]:.6f}   Fluxo = {y_detrend[i]:.6f}")

# ============================================================
# GRÁFICO E LEGENDAS
# ============================================================

fig, ax = plt.subplots(1, 1, figsize=(14, 6))

# --- Plotagem dos Dados ---
ax.plot(t, residual_manchas, 'k.-', ms=1.5, lw=0.5, alpha=0.5, zorder=1,
        label='Residual completo')

# --- Plotagem das Linhas de Referência ---
ax.axhline(y_m, color='green', lw=1.5, ls='-', alpha=0.9, zorder=5, 
           label=f'Média = {y_m:.5f}')
ax.axhline(med2, color='darkorange', lw=1.5, ls='--', alpha=0.9, zorder=5, 
           label=f'+3σ = {med2:.5f}')
ax.axhline(med2_inf, color='darkorange', lw=1.5, ls=':', alpha=0.9, zorder=5, 
           label=f'-3σ = {med2_inf:.5f}')
ax.axhline(y_m + desvio2, color='red', lw=1.5, ls='-.', alpha=0.8, zorder=5, 
           label=f'Média + σ = {y_m + desvio2:.5f}')

# Picos encontrados fora da nova máscara
ax.plot(x_detrend[peaks], y_detrend[peaks], marker='x', linestyle='None', color='deeppink', ms=8, mew=2, zorder=20,
        label=f'Picos > +3σ ({len(peaks)})')

# --- MÁSCARA CSV (flares_transit_1.txt) ---
for i, linha in df_nova_mascara.iterrows():
    ax.axvspan(linha['Inicio'], linha['Fim'], color='purple', alpha=0.20, zorder=0,
               label='Máscara flares_transit_1' if i == 0 else None)

# --- Planeta b ---
for i, tc in enumerate(centers_b):
    ax.axvspan(tc-dur_b/2, tc+dur_b/2, color='royalblue', alpha=0.18, zorder=0,
               label='AU Mic b' if i == 0 else None)
    ax.axvline(tc, color='royalblue', lw=1.0, ls='--', alpha=0.6)

# --- Planeta c ---
for i, tc in enumerate(centers_c):
    ax.axvspan(tc-dur_c/2, tc+dur_c/2, color='seagreen', alpha=0.18, zorder=0,
               label='AU Mic c' if i == 0 else None)
    ax.axvline(tc, color='seagreen', lw=1.0, ls='--', alpha=0.6)

# --- Planeta d ---
for i, tc in enumerate(centers_d):
    ax.axvline(tc, color='purple', lw=1.2, ls='-.', alpha=0.7,
               label='AU Mic d' if i == 0 else None)

# --- Geração da Legenda e Formatação ---
ax.legend(loc='upper right', fontsize=9, framealpha=0.95, ncol=2)
ax.set_xlabel('Tempo - 2457000 [BTJD dias]', fontsize=13)
ax.set_ylabel('Fluxo Residual Normalizado', fontsize=13)
ax.set_title('AU Mic – Residual com Máscara Única, Limiares, Trânsitos e Picos', fontsize=14, fontweight='bold')
ax.grid(alpha=0.25)
ax.set_xlim(t.min(), t.max())

plt.tight_layout()
plt.show()

✅ Arquivo 'flares_transit_1.txt' carregado com sucesso!
Trânsitos AU Mic d: 2
  BTJD = 3893.24906
  BTJD = 3905.98502


UFuncTypeError: ufunc 'less_equal' did not contain a loop with signature matching types (<class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.StrDType'>) -> None

In [12]:
from astropy.stats import sigma_clipped_stats
from scipy.signal import find_peaks
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# CARREGAMENTO DA MÁSCARA (flares_transit_1.txt)
# ============================================================
try:
    df_nova_mascara = pd.read_csv('flares_transit_1.txt')
    print("✅ Arquivo 'flares_transit_1.txt' carregado com sucesso!")
except FileNotFoundError:
    print("⚠️ Arquivo 'flares_transit_1.txt' não encontrado! Criando DataFrame vazio.")
    df_nova_mascara = pd.DataFrame(columns=['Inicio', 'Fim'])

# >>> FIX MÍNIMO (CORREÇÃO DO ERRO DE TIPO) <<<
df_nova_mascara['Inicio'] = pd.to_numeric(df_nova_mascara['Inicio'], errors='coerce')
df_nova_mascara['Fim'] = pd.to_numeric(df_nova_mascara['Fim'], errors='coerce')
df_nova_mascara = df_nova_mascara.dropna()

# ============================================================
# PARÂMETROS AU Mic d E CÁLCULO DOS TRÂNSITOS
# ============================================================
T0_d   = 2458333.3211 - 2457000
P_d    = 12.73596
dur_d  = None

def transit_times(T0, P, t_min, t_max):
    n_min = int(np.ceil((t_min - T0) / P))
    n_max = int(np.floor((t_max - T0) / P))
    return [T0 + n * P for n in range(n_min, n_max + 1)]

# Assumindo que o array 't' já existe
centers_d = transit_times(T0_d, P_d, t.min(), t.max())

print(f"Trânsitos AU Mic d: {len(centers_d)}")
for tc in centers_d:
    print(f"  BTJD = {tc:.5f}")

# ============================================================
# FILTRAGEM DOS PONTOS BONS (Apenas com a máscara do TXT)
# ============================================================
x_detrend = []
y_detrend = []

for j in range(len(t)):
    tempo_atual = t[j]
    is_good = True
    
    for _, linha in df_nova_mascara.iterrows():
        if linha['Inicio'] <= tempo_atual <= linha['Fim']:
            is_good = False
            break
                
    if is_good:
        y_detrend.append(residual_manchas[j])
        x_detrend.append(t[j])

x_detrend = np.array(x_detrend)
y_detrend = np.array(y_detrend)

print(f"\nPontos totais:     {len(t)}")
print(f"Pontos bons:       {len(y_detrend)}")
print(f"Pontos mascarados: {len(t)-len(y_detrend)}")

# ============================================================
# MÉDIA E DESVIO INICIAIS
# ============================================================
y_mean = np.mean(y_detrend)
desvio = np.std(y_detrend)

med_desv2 = y_mean + 2 * desvio
med_desv3 = y_mean - 2 * desvio

print(f"\n--- 1ª iteração ---")
print(f"Média:                {y_mean:.6f}")
print(f"Desvio padrão:        {desvio:.6f}")
print(f"Corte superior (+2σ): {med_desv2:.6f}")
print(f"Corte inferior (-2σ): {med_desv3:.6f}")

# ============================================================
# CORTE ±2σ
# ============================================================
xdefinitivo = []
ydefinitivo = []

for j in range(len(y_detrend)):
    if med_desv3 < y_detrend[j] < med_desv2:
        ydefinitivo.append(y_detrend[j])
        xdefinitivo.append(x_detrend[j])

xdefinitivo = np.array(xdefinitivo)
ydefinitivo = np.array(ydefinitivo)

print(f"\nPontos após corte ±2σ: {len(ydefinitivo)}")

# ============================================================
# SIGMA CLIPPING
# ============================================================
y_m, _, desvio2 = sigma_clipped_stats(ydefinitivo, sigma=3, maxiters=5)

med2 = y_m + 3 * desvio2
med2_inf = y_m - 3 * desvio2

print(f"\n--- 2ª iteração (sigma-clipping 3σ iterativo) ---")
print(f"Média clipped:  {y_m:.6f}")
print(f"Desvio padrão:  {desvio2:.6f}")
print(f"Limiar +3σ:     {med2:.6f}")
print(f"Limiar -3σ:     {med2_inf:.6f}")

# ============================================================
# DETECÇÃO DOS PICOS
# ============================================================
peaks, properties = find_peaks(y_detrend, height=med2)

print(f"\nNúmero de picos encontrados: {len(peaks)}")
for i in peaks:
    print(f"Tempo = {x_detrend[i]:.6f}   Fluxo = {y_detrend[i]:.6f}")

# ============================================================
# GRÁFICO E LEGENDAS
# ============================================================
fig, ax = plt.subplots(1, 1, figsize=(14, 6))

ax.plot(t, residual_manchas, 'k.-', ms=1.5, lw=0.5, alpha=0.5, zorder=1,
        label='Residual completo')

ax.axhline(y_m, color='green', lw=1.5, ls='-', alpha=0.9, zorder=5, 
           label=f'Média = {y_m:.5f}')
ax.axhline(med2, color='darkorange', lw=1.5, ls='--', alpha=0.9, zorder=5, 
           label=f'+3σ = {med2:.5f}')
ax.axhline(med2_inf, color='darkorange', lw=1.5, ls=':', alpha=0.9, zorder=5, 
           label=f'-3σ = {med2_inf:.5f}')
ax.axhline(y_m + desvio2, color='red', lw=1.5, ls='-.', alpha=0.8, zorder=5, 
           label=f'Média + σ = {y_m + desvio2:.5f}')

ax.plot(x_detrend[peaks], y_detrend[peaks], marker='x', linestyle='None',
        color='deeppink', ms=8, mew=2, zorder=20,
        label=f'Picos > +3σ ({len(peaks)})')

# --- MÁSCARA CSV ---
for i, linha in df_nova_mascara.iterrows():
    ax.axvspan(linha['Inicio'], linha['Fim'], color='purple', alpha=0.20, zorder=0,
               label='Máscara flares_transit_1' if i == 0 else None)

ax.legend(loc='upper right', fontsize=9, framealpha=0.95, ncol=2)
ax.set_xlabel('Tempo - 2457000 [BTJD dias]', fontsize=13)
ax.set_ylabel('Fluxo Residual Normalizado', fontsize=13)
ax.set_title('AU Mic – Residual com Máscara Única, Limiares, Trânsitos e Picos',
             fontsize=14, fontweight='bold')
ax.grid(alpha=0.25)
ax.set_xlim(t.min(), t.max())

plt.tight_layout()
plt.show()

✅ Arquivo 'flares_transit_1.txt' carregado com sucesso!
Trânsitos AU Mic d: 2
  BTJD = 3893.24906
  BTJD = 3905.98502

Pontos totais:     15111
Pontos bons:       13251
Pontos mascarados: 1860

--- 1ª iteração ---
Média:                0.999993
Desvio padrão:        0.000614
Corte superior (+2σ): 1.001221
Corte inferior (-2σ): 0.998766

Pontos após corte ±2σ: 12640

--- 2ª iteração (sigma-clipping 3σ iterativo) ---
Média clipped:  1.000007
Desvio padrão:  0.000434
Limiar +3σ:     1.001309
Limiar -3σ:     0.998704

Número de picos encontrados: 166
Tempo = 3883.455401   Fluxo = 1.001387
Tempo = 3883.945682   Fluxo = 1.002498
Tempo = 3883.948459   Fluxo = 1.001711
Tempo = 3883.951237   Fluxo = 1.001522
Tempo = 3883.954015   Fluxo = 1.001396
Tempo = 3883.965126   Fluxo = 1.001357
Tempo = 3884.188738   Fluxo = 1.001345
Tempo = 3884.233183   Fluxo = 1.001499
Tempo = 3884.237350   Fluxo = 1.001823
Tempo = 3884.254016   Fluxo = 1.001345
Tempo = 3884.669295   Fluxo = 1.001423
Tempo = 3884.68596

In [17]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.integrate import simpson
from PyAstronomy.pyasl import planck

# =====================================================
# LEITURA DAS FLARES
# =====================================================

flares = []

with open("flares_transit_1.txt") as f:

    next(f)  # pula cabeçalho

    for linha in f:

        linha = linha.strip()

        if not linha:
            continue

        transito = "#transito" in linha.lower()

        linha = linha.split("#")[0].strip()

        inicio, fim = linha.split(",")

        flares.append({
            "Inicio": float(inicio),
            "Fim": float(fim),
            "Transito": transito
        })

flares = pd.DataFrame(flares)

print(f"Número de flares encontradas: {len(flares)}")

# =====================================================
# FUNÇÃO DE RESPOSTA DO TESS
# =====================================================

tess_f = pd.read_csv("tess_functionx1.csv")

nm = np.array(tess_f["nm"])
ry = np.array(tess_f["ry"])

nm_metros = nm * 1e-9
nm_cm = nm * 1e-7

# =====================================================
# PARÂMETROS ESTELARES
# =====================================================

Teff = 3700
Tflare = 9000

raio = 0.75 * 6.957e10      # cm

sb1 = 5.67051e-5            # erg cm^-2 s^-1 K^-4

# =====================================================
# PLANCK
# =====================================================

Bs = planck(Teff, lam=nm_metros) * 1e-7
Bf = planck(Tflare, lam=nm_metros) * 1e-7

fator_area = (
    simpson(ry * Bs, x=nm_cm)
    /
    simpson(ry * Bf, x=nm_cm)
)

# =====================================================
# FUNÇÃO PARA EXTRAIR FLARE
# =====================================================

def extrair_flare(start, stop, tempo, fluxo):

    mask = (tempo >= start) & (tempo <= stop)

    return (
        np.array(fluxo[mask]),
        np.array(tempo[mask])
    )

# =====================================================
# CÁLCULO DAS ENERGIAS
# =====================================================

lista_flares = []

for i, row in flares.iterrows():

    start = row["Inicio"]
    stop = row["Fim"]

    if stop <= start:
        continue

    flux_flare, tempo_flare = extrair_flare(
        start,
        stop,
        t,
        residual_manchas
    )

    if len(flux_flare) < 3:
        continue

    amplitude_flare = flux_flare - 1

    Aflare = amplitude_flare * np.pi * raio**2

    Atotal = Aflare * fator_area

    L = sb1 * (Tflare**4) * Atotal

    energia = simpson(
        L,
        x=tempo_flare * 86400
    )

    tempo_plot = (
        tempo_flare - tempo_flare[0]
    ) * 24 * 60

    lista_flares.append({
        "tempo": tempo_plot,
        "fluxo": amplitude_flare,
        "energia": energia,
        "transito": row["Transito"]
    })

print(f"Flares válidas: {len(lista_flares)}")

# =====================================================
# FUNÇÃO PARA PLOTAR MOSAICO
# =====================================================

# =====================================================
# FUNÇÃO PARA PLOTAR MOSAICO
# =====================================================

# =====================================================
# FUNÇÃO PARA PLOTAR MOSAICO
# =====================================================

def plotar_mosaico(flares_plot, titulo):

    ncols = 4
    nrows = 5

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(14, 12)
    )

    axes = axes.flatten()

    for i, flare in enumerate(flares_plot):

        ax = axes[i]

        # Define a cor: vermelho se tiver trânsito, preto ("k") se não tiver
        cor = "red" if flare["transito"] else "blue"

        ax.plot(
            flare["tempo"],
            flare["fluxo"],
            marker=".",        # Adiciona os pontos do estilo ".-"
            linestyle="-",     # Adiciona a linha contínua do estilo ".-"
            color=cor,         # Aplica a cor condicional (preto ou vermelho)
            lw=1.5
        )

        ax.set_title(
            f"Flare {i+1}\nE={flare['energia']:.2e}",
            fontsize=8
        )

        ax.tick_params(
            axis="both",
            labelsize=6
        )

        ax.grid(alpha=0.3)

    # remove painéis vazios
    for j in range(len(flares_plot), len(axes)):
        fig.delaxes(axes[j])

    fig.suptitle(
        titulo,
        fontsize=16
    )

    plt.tight_layout()
    plt.show()

# =====================================================
# PRIMEIRAS 20 FLARES
# =====================================================

plotar_mosaico(
    lista_flares[:20],
    "Flares 1–20"
)

# =====================================================
# FLARES 21–40
# =====================================================

if len(lista_flares) > 20:

    plotar_mosaico(
        lista_flares[20:40],
        "Flares 21–40"
    )

Número de flares encontradas: 41
Flares válidas: 40


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.integrate import simpson
from PyAstronomy.pyasl import planck

# =====================================================
# LEITURA DAS FLARES (IGNORANDO TRÂNSITOS)
# =====================================================

flares = []

with open("flares_transit_1.txt") as f:

    next(f)  # pula cabeçalho

    for linha in f:

        linha = linha.strip()

        if not linha:
            continue

        # Se for um trânsito, ignora e vai para a próxima linha
        if "#transito" in linha.lower():
            continue

        linha = linha.split("#")[0].strip()

        inicio, fim = linha.split(",")

        flares.append({
            "Inicio": float(inicio),
            "Fim": float(fim),
            "Transito": False  # Como excluímos os trânsitos, todos aqui serão False
        })

flares = pd.DataFrame(flares)

print(f"Número de flares puras encontradas: {len(flares)}")

# =====================================================
# FUNÇÃO DE RESPOSTA DO TESS
# =====================================================

tess_f = pd.read_csv("tess_functionx1.csv")

nm = np.array(tess_f["nm"])
ry = np.array(tess_f["ry"])

nm_metros = nm * 1e-9
nm_cm = nm * 1e-7

# =====================================================
# PARÂMETROS ESTELARES
# =====================================================

Teff = 3700
Tflare = 9000

raio = 0.75 * 6.957e10      # cm

sb1 = 5.67051e-5            # erg cm^-2 s^-1 K^-4

# =====================================================
# PLANCK
# =====================================================

Bs = planck(Teff, lam=nm_metros) * 1e-7
Bf = planck(Tflare, lam=nm_metros) * 1e-7

fator_area = (
    simpson(ry * Bs, x=nm_cm)
    /
    simpson(ry * Bf, x=nm_cm)
)

# =====================================================
# FUNÇÃO PARA EXTRAIR FLARE
# =====================================================

def extrair_flare(start, stop, tempo, fluxo):

    mask = (tempo >= start) & (tempo <= stop)

    return (
        np.array(fluxo[mask]),
        np.array(tempo[mask])
    )

# =====================================================
# CÁLCULO DAS ENERGIAS
# =====================================================

lista_flares = []

for i, row in flares.iterrows():

    start = row["Inicio"]
    stop = row["Fim"]

    if stop <= start:
        continue

    flux_flare, tempo_flare = extrair_flare(
        start,
        stop,
        t,                  # Certifique-se de que 't' está definido no seu ambiente
        residual_manchas    # Certifique-se de que 'residual_manchas' está definido
    )

    if len(flux_flare) < 3:
        continue

    amplitude_flare = flux_flare - 1

    Aflare = amplitude_flare * np.pi * raio**2

    Atotal = Aflare * fator_area

    L = sb1 * (Tflare**4) * Atotal

    energia = simpson(
        L,
        x=tempo_flare * 86400
    )

    tempo_plot = (
        tempo_flare - tempo_flare[0]
    ) * 24 * 60

    lista_flares.append({
        "tempo": tempo_plot,
        "fluxo": amplitude_flare,
        "energia": energia,
        "transito": row["Transito"]
    })

print(f"Flares válidas para plotagem: {len(lista_flares)}")
# ADICIONE ESTAS LINHAS NO SEU CÓDIGO ORIGINAL:

# =====================================================
# FUNÇÃO PARA PLOTAR MOSAICO
# =====================================================

# =====================================================
# FUNÇÃO PARA PLOTAR MOSAICO
# =====================================================

def plotar_mosaico(flares_plot, titulo):

    ncols = 4
    nrows = 5

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(14, 12)
    )

    axes = axes.flatten()

    for i, flare in enumerate(flares_plot):

        ax = axes[i]

        ax.plot(
            flare["tempo"],
            flare["fluxo"],
            "b.-",
            lw=1.5
        )

        ax.set_title(
            f"Flare {i+1}\nE={flare['energia']:.2e}",
            fontsize=8
        )

        # =====================================================
        # ADICIONANDO OS NOMES DOS EIXOS AQUI
        # =====================================================
        ax.set_xlabel("Tempo (minutos)", fontsize=7)
        ax.set_ylabel("Fluxo Relativo", fontsize=7)

        ax.tick_params(
            axis="both",
            labelsize=6
        )

        ax.grid(alpha=0.3)

    # remove painéis vazios
    for j in range(len(flares_plot), len(axes)):
        fig.delaxes(axes[j])

    fig.suptitle(
        titulo,
        fontsize=16
    )

    plt.tight_layout()
    plt.show()

# =====================================================
# PRIMEIRAS 20 FLARES
# =====================================================

if len(lista_flares) > 0:
    plotar_mosaico(
        lista_flares[:20],
        "Flares 1–20 (Apenas Flares Puras)"
    )

# =====================================================
# FLARES 21–40
# =====================================================

if len(lista_flares) > 20:
    plotar_mosaico(
        lista_flares[20:40],
        "Flares 21–40 "
    )

Número de flares puras encontradas: 38
Flares válidas para plotagem: 38


In [21]:
# =====================================================
# CÁLCULO DAS MÉDIAS (ENERGIA E DURAÇÃO)
# =====================================================

energias = []
duracoes = []

# Extraindo os dados de todas as flares válidas
for flare in lista_flares:
    energias.append(flare["energia"])
    
    # A duração é o último valor do array de tempo, já que ele começa em 0
    duracao_total = flare["tempo"][-1]
    duracoes.append(duracao_total)

# Convertendo para arrays do numpy para facilitar os cálculos
energias = np.array(energias)
duracoes = np.array(duracoes)

# Calculando as médias
energia_media = np.mean(energias)
duracao_media = np.mean(duracoes)

print(f"--- RESUMO DAS FLARES ---")
print(f"Total de flares analisadas: {len(lista_flares)}")
print(f"Duração média: {duracao_media:.2f} minutos")
print(f"Energia média:  {energia_media:.2e} erg")
print("-" * 25)

# =====================================================
# PLOTANDO OS HISTOGRAMAS
# =====================================================

# Criando a figura com 1 linha e 2 colunas
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# -----------------------------------------------------
# Histograma 1: DURAÇÃO
# -----------------------------------------------------
ax1.hist(duracoes, bins=15, color='gray', edgecolor='black', alpha=0.8)
ax1.set_title("Distribuição da Duração das Flares", fontsize=14)
ax1.set_xlabel("Duração (minutos)", fontsize=12)
ax1.set_ylabel("Frequência (Número de Flares)", fontsize=12)
ax1.grid(alpha=0.3)

# -----------------------------------------------------
# Histograma 2: ENERGIA (em Log)
# -----------------------------------------------------
# Usamos np.log10 porque as energias variam em ordens de grandeza muito altas
log_energias = np.log10(energias)

ax2.hist(log_energias, bins=15, color='orange', edgecolor='black', alpha=0.8)
ax2.set_title("Distribuição da Energia das Flares", fontsize=14)
ax2.set_xlabel("Log da Energia [erg]", fontsize=12)
ax2.set_ylabel("Frequência (Número de Flares)", fontsize=12)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

--- RESUMO DAS FLARES ---
Total de flares analisadas: 38
Duração média: 100.63 minutos
Energia média:  2.07e+33 erg
-------------------------


In [32]:
# =====================================================
# CÁLCULO DAS MÉDIAS (ENERGIA E DURAÇÃO)
# =====================================================

energias = []
duracoes = []

for flare in lista_flares:
    energias.append(flare["energia"])
    duracao_total = flare["tempo"][-1]
    duracoes.append(duracao_total)

energias = np.array(energias)
duracoes = np.array(duracoes)

energia_media = np.mean(energias)
duracao_media = np.mean(duracoes)

print(f"--- RESUMO DAS EXPLOSÕES ---")
print(f"Total de explosões analisadas: {len(lista_flares)}")
print(f"Duração média: {duracao_media:.2f} minutos")
print(f"Energia média:  {energia_media:.2e} erg")
print("-" * 25)

# =====================================================
# PLOTANDO OS HISTOGRAMAS COM LIMITES
# =====================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# -----------------------------------------------------
# Histograma 1: DURAÇÃO
# -----------------------------------------------------
ax1.hist(duracoes, bins=20, range=(0, 400), color='gray', edgecolor='black', alpha=0.8)

ax1.set_title("Distribuição da Duração das Explosões", fontsize=14)
ax1.set_xlabel("Duração (minutos)", fontsize=12)

# ---> EIXO Y DO PRIMEIRO GRÁFICO
ax1.set_ylabel("Quantidade de Explosões", fontsize=12)

ax1.set_xlim(0, 400)
ax1.set_ylim(0, 30)
ax1.grid(alpha=0.3)

# -----------------------------------------------------
# Histograma 2: ENERGIA (em Log)
# -----------------------------------------------------
log_energias = np.log10(energias)

ax2.hist(log_energias, bins=15, range=(31.5, 34.5), color='orange', edgecolor='black', alpha=0.8)

ax2.set_title("Distribuição da Energia das Explosões", fontsize=14)
ax2.set_xlabel("Log da Energia [erg]", fontsize=12)

# ---> EIXO Y DO SEGUNDO GRÁFICO
ax2.set_ylabel("Quantidade de Explosões", fontsize=12)

ax2.set_xlim(31.5, 34.5)
ax2.grid(alpha=0.3)


print("\n=== COPIE OS VETORES ABAIXO ===")
print(f"energias = np.array({list(energias)})")
print(f"duracoes = np.array({list(duracoes)})")
print("===============================\n")

plt.tight_layout()
plt.show()

--- RESUMO DAS EXPLOSÕES ---
Total de explosões analisadas: 38
Duração média: 100.63 minutos
Energia média:  2.07e+33 erg
-------------------------

=== COPIE OS VETORES ABAIXO ===
energias = np.array([np.float64(3.679600176244424e+33), np.float64(2.1558332245336452e+33), np.float64(1.3829863789751324e+33), np.float64(1.4751445324560855e+33), np.float64(2.7155018319775864e+32), np.float64(3.2396125764736046e+32), np.float64(5.159608475220695e+32), np.float64(3.381273488369089e+32), np.float64(1.0006303332894954e+33), np.float64(2.2902903129949134e+32), np.float64(2.5849616007271504e+32), np.float64(3.91205073809735e+32), np.float64(2.2349097169646696e+32), np.float64(7.39633470492188e+32), np.float64(5.528150974781433e+32), np.float64(9.977556335077186e+32), np.float64(1.0739037938617131e+33), np.float64(8.413993076745618e+32), np.float64(1.2745821482624827e+33), np.float64(5.3365951814804466e+32), np.float64(6.347514299128405e+32), np.float64(7.038524673264005e+32), np.float64(8.49446

In [31]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.timeseries import LombScargle

# Tempo e fluxo
t = np.array(lc2_m.time.value)
f = np.array(lc2_m.flux)

# Remove a média
f = f - np.mean(f)

# Intervalo de períodos (dias)
period = np.linspace(3, 6, 10000)
frequency = 1 / period

# Lomb-Scargle
ls = LombScargle(t, f)
power = ls.power(frequency)

# Melhor período
best_period = period[np.argmax(power)]
print(f'Período = {best_period:.6f} dias')

# Gráfico
plt.figure(figsize=(10,5))
plt.plot(period, power, lw=1)
plt.axvline(best_period, color='red', linestyle='--',
            label=f'P = {best_period:.4f} dias')

plt.xlabel('Período (dias)', fontsize=14)
plt.ylabel('Potência', fontsize=14)
plt.title('Periodograma de Lomb-Scargle', fontsize=15)
plt.legend()
plt.grid(alpha=0.3)

plt.show()

Período = 4.865887 dias


In [33]:
import numpy as np
from astropy.modeling.models import Gaussian2D


def FFD(ED, TOTEXP=1., Lum=30., fluxerr=0., dur=[], logY=True, est_comp=False):
    '''
    Given a set of stellar flares, with accompanying durations light curve properties,
    compute the reverse cumulative Flare Frequency Distribution (FFD), and
    approximate uncertainties in both energy and rate (X,Y) dimensions.
    This diagram can be read as measuring the number of flares per day at a
    given energy or larger.

    Not a complicated task, just tedious.

    Y-errors (rate) are computed using Poisson upper-limit approximation from
    Gehrels (1986) "Confidence limits for small numbers of events in astrophysical data", https://doi.org/10.1086/164079
    Eqn 7, assuming S=1.

    X-errors (event energy) are computed following Signal-to-Noise approach commonly
    used for Equivalent Widths in spectroscopy, from
    Vollmann & Eversberg (2006) "Astronomische Nachrichten, Vol.327, Issue 9, p.862", https://dx.doi.org/10.1002/asna.200610645
    Eqn 6.

    Parameters
    ----------
    ED : array of Equivalent Durations of flares (units of seconds)
    TOTEXP : total duration of observations (units of days)
    Lum : log luminosity of the star (units of log10[erg/s])
    fluxerr : the average flux errors of your data (in relative flux units)
    dur : array of flare durations (units of days)
    logY : if True return Y-axis (and error) in log rate (Default: True)
    est_comp : estimate incompleteness using histogram method, scale Y errors?
        (Default: True)

    Returns
    -------
    ffd_x, ffd_y, ffd_xerr, ffd_yerr

    X coordinate always assumed to be log_10(Energy)
    Y coordinate is log_10(N/Day) by default, but optionally is N/Day

    Upgrade Ideas
    -------------
    - More graceful behavior if only an array of flares and a total duration are
        specified (i.e. just enough to make ffd_x, ffd_y)
    - Better propogation of specific flux errors in the light curve, rather than
        average error used
    - Include detrending errors? (e.g. from a GP)
    - Asymmetric Poisson errors?
    - Better handling of incompleteness?

    '''
    # REVERSE sort the flares in energy
    ss = np.argsort(np.array(ED))[::-1]
    ffd_x = np.log10(ED[ss]) + Lum

    Num = np.arange(1, len(ffd_x)+1)
    ffd_y = Num / TOTEXP

    # approximate the Poisson Y errors using Gehrels (1986) eqn 7
    Perror = np.sqrt(Num + 0.75) + 1.0
    ffd_yerr = Perror / TOTEXP

    # estimate completeness using the cumulative distribution of the histogram
    if est_comp:
        # make very loose guess at how many bins to choose
        nbin = int(np.sqrt(len(ffd_x)))
        if nbin < 10:
            nbin=10 # but use at least 10 bins

        # make histogram of the log(energies)
        hh, be = np.histogram(ffd_x, bins=nbin, range=[np.nanmin(ffd_x), np.nanmax(ffd_x)])
        hh = hh/np.nanmax(hh)
        # make cumulative distribution of the histogram, scale to =1 at the hist peak
        cc = np.cumsum(hh)/np.sum(hh[0:np.argmax(hh)])
        be = (be[1:]+be[0:-1])/2
        # make completeness = 1 for energies above the histogram peak
        cc[np.argmax(hh):] = 1
        # interpolate the cumulative histogram curve back to the original energies
        ycomp = np.interp(ffd_x, be, cc)
        # scale the y-errors by the completeness factor (i.e. inflate small energy errors)
        ffd_yerr = ffd_yerr / ycomp

    if logY:
        # transform FFD Y and Y Error into log10
        ffd_yerr = np.abs(ffd_yerr / np.log(10.) / ffd_y)
        ffd_y = np.log10(ffd_y)

    # compute X uncertainties for FFD
    if len(dur)==len(ffd_x):

        # assume relative flux error = 1/SN
        S2N = 1/fluxerr
        # based on Equivalent Width error
        # Eqn 6, Vollmann & Eversberg (2006) Astronomische Nachrichten, Vol.327, Issue 9, p.862
        ED_err = np.sqrt(2)*(dur[ss]*86400. - ED[ss])/S2N
        ffd_xerr = np.abs((ED_err) / np.log(10.) / ED[ss]) # convert to log
    else:
        # not particularly meaningful, but an interesting shape. NOT reccomended
        print('Warning: Durations not set. Making bad assumptions about the FFD X Error!')
        ffd_xerr = (1/np.sqrt(ffd_x-np.nanmin(xT))/(np.nanmax(ffd_x)-np.nanmin(ffd_x)))

    return ffd_x, ffd_y, ffd_xerr, ffd_yerr


def FlareKernel(x, y, xe, ye, Nx=100, Ny=100, xlim=[], ylim=[], return_axis=True):
    '''
    Use 2D Gaussians (from astropy models) to make a basic kernel density,
    with errors in both X and Y considered. Turn into a 2D "image"

    Upgrade Ideas
    -------------
    It's slow. Since Gaussians are defined analytically, maybe this could be
    re-cast as a single array math opperation, and then refactored to have the
    same fit/evaluate behavior as KDE functions.  Hmm...
    '''

    if len(xlim) == 0:
        xlim = [np.nanmin(x) - np.nanmean(xe), np.nanmax(x) + np.nanmean(xe)]
    if len(ylim) == 0:
        ylim = [np.nanmin(y) - np.nanmean(ye), np.nanmax(y) + np.nanmean(ye)]

    xx,yy = np.meshgrid(np.linspace(xlim[0], xlim[1], Nx),
                        np.linspace(ylim[0], ylim[1], Ny), indexing='xy')
    dx = (np.max(xlim)-np.min(xlim)) / (Nx-1)
    dy = (np.max(ylim)-np.min(ylim)) / (Ny-1)

    im = np.zeros_like(xx)

    for k in range(len(x)):
        g = Gaussian2D(amplitude=1/(2*np.pi*(xe[k]+dx)*(ye[k]+dy)),
                       x_mean=x[k], y_mean=y[k], x_stddev=xe[k]+dx, y_stddev=ye[k]+dy)
        tmp = g(xx,yy)
        if np.isfinite(np.sum(tmp)):
            im = im + tmp

    if return_axis:
        return im, xx, yy
    else:
        return im

In [42]:
ener = np.array([np.float64(3.679600176244424e+33), np.float64(2.1558332245336452e+33), np.float64(1.3829863789751324e+33), np.float64(1.4751445324560855e+33), np.float64(2.7155018319775864e+32), np.float64(3.2396125764736046e+32), np.float64(5.159608475220695e+32), np.float64(3.381273488369089e+32), np.float64(1.0006303332894954e+33), np.float64(2.2902903129949134e+32), np.float64(2.5849616007271504e+32), np.float64(3.91205073809735e+32), np.float64(2.2349097169646696e+32), np.float64(7.39633470492188e+32), np.float64(5.528150974781433e+32), np.float64(9.977556335077186e+32), np.float64(1.0739037938617131e+33), np.float64(8.413993076745618e+32), np.float64(1.2745821482624827e+33), np.float64(5.3365951814804466e+32), np.float64(6.347514299128405e+32), np.float64(7.038524673264005e+32), np.float64(8.494465077840076e+33), np.float64(1.4542712952470802e+34), np.float64(7.030763022849664e+32), np.float64(7.764221747508636e+32), np.float64(1.228412051606988e+34), np.float64(6.789277300253199e+33), np.float64(3.334958212050767e+33), np.float64(4.976951681984643e+32), np.float64(1.2832008563082165e+33), np.float64(2.6317980784071556e+32), np.float64(4.5711103178543554e+32), np.float64(3.813189093659259e+32), np.float64(3.37316601984778e+32), np.float64(1.7790376366203564e+33), np.float64(4.9821866344919374e+32), np.float64(6.785744692158493e+33)])
dur = np.array([np.float64(146.00095413610688), np.float64(180.00045575747208), np.float64(98.00018113935948), np.float64(108.0000435882539), np.float64(27.99999935523374), np.float64(31.99987137923017), np.float64(41.999743217165815), np.float64(39.99968542266288), np.float64(113.99898600546294), np.float64(57.99946705090406), np.float64(29.99968678683217), np.float64(55.99934595404193), np.float64(63.99911821630667), np.float64(89.9987928068731), np.float64(75.99884567724075), np.float64(89.99860840369365), np.float64(87.99861672276165), np.float64(85.9985090389091), np.float64(95.9982957795728), np.float64(37.99931085508433), np.float64(69.99867952770728), np.float64(105.99797255228623), np.float64(235.99633076344617), np.float64(323.99417551554507), np.float64(37.99916623451281), np.float64(57.99866808592924), np.float64(145.9963747049187), np.float64(129.9966342861444), np.float64(113.99699445086299), np.float64(85.99748009284667), np.float64(289.9912199116079), np.float64(43.9986272695387), np.float64(75.9975027955079), np.float64(51.998266858718125), np.float64(47.99839475941553), np.float64(145.99485277984058), np.float64(89.99677216248529), np.float64(209.99216998512566)])


In [46]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.modeling.models import Gaussian2D

# ==============================================================================
# 1. FUNÇÕES HISTÓRICAS DO FFD E KERNEL (CORRIGIDAS)
# ==============================================================================

def FFD(ED, TOTEXP=1., Lum=30., fluxerr=0., dur=[], logY=True, est_comp=False):
    '''
    Given a set of stellar flares, compute the reverse cumulative Flare Frequency Distribution (FFD).
    '''
    # REVERSE sort the flares in energy
    ss = np.argsort(np.array(ED))[::-1]
    ffd_x = np.log10(ED[ss]) + Lum

    Num = np.arange(1, len(ffd_x)+1)
    ffd_y = Num / TOTEXP

    # approximate the Poisson Y errors using Gehrels (1986) eqn 7
    Perror = np.sqrt(Num + 0.75) + 1.0
    ffd_yerr = Perror / TOTEXP

    # estimate completeness using the cumulative distribution of the histogram
    if est_comp:
        nbin = int(np.sqrt(len(ffd_x)))
        if nbin < 10:
            nbin = 10

        hh, be = np.histogram(ffd_x, bins=nbin, range=[np.nanmin(ffd_x), np.nanmax(ffd_x)])
        hh = hh / np.nanmax(hh)
        cc = np.cumsum(hh) / np.sum(hh[0:np.argmax(hh)])
        be = (be[1:] + be[0:-1]) / 2
        cc[np.argmax(hh):] = 1
        ycomp = np.interp(ffd_x, be, cc)
        
        # Previne divisão por zero se a completude for mapeada como zero
        ycomp = np.where(ycomp == 0, 1e-4, ycomp)
        ffd_yerr = ffd_yerr / ycomp

    if logY:
        ffd_yerr = np.abs(ffd_yerr / np.log(10.) / ffd_y)
        ffd_y = np.log10(ffd_y)

    # compute X uncertainties for FFD
    if len(dur) == len(ffd_x):
        S2N = 1 / fluxerr
        # Eqn 6, Vollmann & Eversberg (2006)
        ED_err = np.sqrt(2) * (dur[ss] * 86400. - ED[ss]) / S2N
        ffd_xerr = np.abs((ED_err) / np.log(10.) / ED[ss])
    else:
        print('Warning: Durations not set properly. Making bad assumptions about X Error!')
        # Evita erro de variável não definida 'xT' que havia no código original
        ffd_xerr = (1 / np.sqrt(ffd_x - np.nanmin(ffd_x) + 1e-4) / (np.nanmax(ffd_x) - np.nanmin(ffd_x)))

    return ffd_x, ffd_y, ffd_xerr, ffd_yerr


def FlareKernel(x, y, xe, ye, Nx=100, Ny=100, xlim=[], ylim=[], return_axis=True):
    '''
    Use 2D Gaussians to make a basic kernel density image.
    '''
    if len(xlim) == 0:
        xlim = [np.nanmin(x) - np.nanmean(xe), np.nanmax(x) + np.nanmean(xe)]
    if len(ylim) == 0:
        ylim = [np.nanmin(y) - np.nanmean(ye), np.nanmax(y) + np.nanmean(ye)]

    xx, yy = np.meshgrid(np.linspace(xlim[0], xlim[1], Nx),
                         np.linspace(ylim[0], ylim[1], Ny), indexing='xy')
    dx = (np.max(xlim) - np.min(xlim)) / (Nx - 1)
    dy = (np.max(ylim) - np.min(ylim)) / (Ny - 1)

    im = np.zeros_like(xx)

    for k in range(len(x)):
        g = Gaussian2D(amplitude=1 / (2 * np.pi * (xe[k] + dx) * (ye[k] + dy)),
                       x_mean=x[k], y_mean=y[k], x_stddev=xe[k] + dx, y_stddev=ye[k] + dy)
        tmp = g(xx, yy)
        if np.isfinite(np.sum(tmp)):
            im = im + tmp

    if return_axis:
        return im, xx, yy
    else:
        return im  # CORRIGIDO: Bug de sintaxe 'return im = s' removido.


# ==============================================================================
# 2. SEUS DADOS E PARAMETRIZAÇÃO (Substitua pelos seus arrays reais)
# ==============================================================================
ener = np.array([np.float64(3.679600176244424e+33), np.float64(2.1558332245336452e+33), np.float64(1.3829863789751324e+33), np.float64(1.4751445324560855e+33), np.float64(2.7155018319775864e+32), np.float64(3.2396125764736046e+32), np.float64(5.159608475220695e+32), np.float64(3.381273488369089e+32), np.float64(1.0006303332894954e+33), np.float64(2.2902903129949134e+32), np.float64(2.5849616007271504e+32), np.float64(3.91205073809735e+32), np.float64(2.2349097169646696e+32), np.float64(7.39633470492188e+32), np.float64(5.528150974781433e+32), np.float64(9.977556335077186e+32), np.float64(1.0739037938617131e+33), np.float64(8.413993076745618e+32), np.float64(1.2745821482624827e+33), np.float64(5.3365951814804466e+32), np.float64(6.347514299128405e+32), np.float64(7.038524673264005e+32), np.float64(8.494465077840076e+33), np.float64(1.4542712952470802e+34), np.float64(7.030763022849664e+32), np.float64(7.764221747508636e+32), np.float64(1.228412051606988e+34), np.float64(6.789277300253199e+33), np.float64(3.334958212050767e+33), np.float64(4.976951681984643e+32), np.float64(1.2832008563082165e+33), np.float64(2.6317980784071556e+32), np.float64(4.5711103178543554e+32), np.float64(3.813189093659259e+32), np.float64(3.37316601984778e+32), np.float64(1.7790376366203564e+33), np.float64(4.9821866344919374e+32), np.float64(6.785744692158493e+33)])
dur = np.array([np.float64(146.00095413610688), np.float64(180.00045575747208), np.float64(98.00018113935948), np.float64(108.0000435882539), np.float64(27.99999935523374), np.float64(31.99987137923017), np.float64(41.999743217165815), np.float64(39.99968542266288), np.float64(113.99898600546294), np.float64(57.99946705090406), np.float64(29.99968678683217), np.float64(55.99934595404193), np.float64(63.99911821630667), np.float64(89.9987928068731), np.float64(75.99884567724075), np.float64(89.99860840369365), np.float64(87.99861672276165), np.float64(85.9985090389091), np.float64(95.9982957795728), np.float64(37.99931085508433), np.float64(69.99867952770728), np.float64(105.99797255228623), np.float64(235.99633076344617), np.float64(323.99417551554507), np.float64(37.99916623451281), np.float64(57.99866808592924), np.float64(145.9963747049187), np.float64(129.9966342861444), np.float64(113.99699445086299), np.float64(85.99748009284667), np.float64(289.9912199116079), np.float64(43.9986272695387), np.float64(75.9975027955079), np.float64(51.998266858718125), np.float64(47.99839475941553), np.float64(145.99485277984058), np.float64(89.99677216248529), np.float64(209.99216998512566)])

# IMPORTANTE: Substitua estes dados fake pelos seus arrays reais: `ener` e `dur`
# Exemplo simulado com 20 flares para o código rodar direto de forma limpa:

# Parâmetros astrofísicos da AU Mic
Lum_star_log = 32.35 
Lum_star_linear = 10**Lum_star_log  # ~ 2.24e32 erg/s
TOTEXP = 50.4                       # Tempo total de observação em dias

# ==============================================================================
# 3. TRATAMENTO, CONVERSÃO DE UNIDADES E EXECUÇÃO
# ==============================================================================
# ==============================================================================
# 3. TRATAMENTO, CONVERSÃO DE UNIDADES E EXECUÇÃO (CORRIGIDO)
# ==============================================================================

# Converter durações de minutos para dias (unidade exigida pelo argumento dur da FFD)
dur_dias = dur / (24 * 60) 

# CORREÇÃO: 'ener' já é a energia total integrada do flare em ergs.
# Não multiplique por dur_segundos, senão você altera a dimensão física dos dados.
energia_total_erg = ener  

# Calcular a Duração Equivalente (ED em segundos) de forma correta
EquivDur = energia_total_erg / Lum_star_linear

# Injetar os parâmetros de ruído calculados no seu Sigma-Clipping (2ª iteração)
flux_quiescente = 1.000007
erro_fotometrico = 0.000434
rel_flux_err = erro_fotometrico / flux_quiescente 

# Executar a computação da distribuição FFD
x, y, xe, ye = FFD(ED=EquivDur, 
                   dur=dur_dias, 
                   Lum=Lum_star_log, 
                   TOTEXP=TOTEXP, 
                   fluxerr=rel_flux_err,
                   est_comp=True)

# Gerar o mapa de densidade (Kernel 2D) baseado nas incertezas corretas
im, xx, yy = FlareKernel(x, y, xe, ye)

# ==============================================================================
# 4. GERAÇÃO DO GRÁFICO FINAL OTIMIZADO
# ==============================================================================

plt.figure(figsize=(9, 6.5))

# Camada de fundo: Contornos suavizados da densidade de probabilidade (Kernel)
contorno = plt.contourf(xx, yy, im, levels=20, cmap='viridis', alpha=0.4)
plt.colorbar(contorno, label='Densidade de Probabilidade (Kernel)')

# Camada da frente: Os dados reais da AU Mic com suas respectivas incertezas
plt.errorbar(x, y, xerr=xe, yerr=ye, fmt='o', color='black', 
             ecolor='gray', elinewidth=1, capsize=2, markersize=5, label='Flares de AU Mic')

# Configurações estéticas e legendas dos eixos
plt.xlabel(r'$\log_{10}$ Energia (erg)', fontsize=12)
plt.ylabel(r'$\log_{10}$ Taxa de Flares ($\text{dia}^{-1}$)', fontsize=12)
plt.title('Flare Frequency Distribution (FFD) - AU Mic', fontsize=14, fontweight='bold')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='upper right')

plt.show()